# Robust Schema Wording for GLiNER2

Paraphrase-ensembled zero-shot intent classification on Banking77.

This notebook runs the main experiment for deterministic schema wording variants. It uses local GLiNER2 inference only, does not fine-tune the model, does not call paid APIs, and does not create manual annotations.

In [ ]:
PROJECT_NAME = "gliner2_schema_wording"
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "mteb/banking77"
MODE = "small"  # smoke / small / full
USE_GPU_IF_AVAILABLE = True
SEED = 42
SMOKE_N_EXAMPLES = 10
SMALL_PER_LABEL = 5
FULL_USE_ALL_TEST = True
MARGIN_THRESHOLD = 0.05
FORCE_RERUN = False
CONFIRM_FULL_RUN = False
OUTPUT_DIR = "/content/gliner2_schema_outputs"

if MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "MODE='full' requires CONFIRM_FULL_RUN=True. This prevents accidental long runs."
    )

## Install dependencies

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()
    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    repo_url = "https://github.com/daidai-su/GLiNER2-demo.git"
    clone_dir = Path("/content/GLiNER2-demo")
    print("Project files were not found in the runtime.")
    print(f"Trying to clone project files from {repo_url} ...")
    try:
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
        PROJECT_ROOT = find_project_root()
    except Exception as exc:
        print(f"Automatic clone failed: {exc}")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Make sure requirements-colab.txt and src/ "
        "are available in the Colab runtime. For a private repo, upload or clone "
        "the full repository into the runtime before running this notebook."
    )

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
)
print("Dependencies installed.")

## Environment check

In [ ]:
import hashlib
import inspect
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from gliner2_project.analysis import (
    compare_method_examples,
    confidence_analysis_frame,
    confusion_pairs_frame,
    method_metrics_frame,
    paired_outcome_frame,
    paired_summary_frame,
    per_label_delta_frame,
    per_label_metrics_frame,
    schema_disagreement_frame,
)
from gliner2_project.data_utils import ensure_label_text_column, stratified_subset, unique_label_texts
from gliner2_project.ensemble import (
    CONFIDENCE_MARGIN_FALLBACK,
    MEAN_CONFIDENCE_ENSEMBLE,
    VOTE_ENSEMBLE,
    confidence_coverage,
    confidence_margin_fallback,
    majority_vote_ensemble,
    mean_confidence_ensemble,
)
from gliner2_project.env_utils import collect_environment_info, print_environment_info
from gliner2_project.gliner2_wrappers import load_gliner2_model, predict_one_classification
from gliner2_project.plotting import (
    plot_confusion_matrix_for_pairs,
    plot_metric_bar,
    plot_schema_disagreement_histogram,
    plot_top_label_improvements,
)
from gliner2_project.schema_variants import (
    BANKING_REQUEST_LABEL,
    CUSTOMER_INTENT_LABEL,
    ENSEMBLE_INPUT_METHODS,
    PLAIN_LABEL,
    QUERY_ABOUT_LABEL,
    RAW_LABEL,
    SINGLE_SCHEMA_METHODS,
    build_all_schema_variants,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

environment_info = print_environment_info()
RUN_START_UTC = datetime.now(timezone.utc).isoformat()
print(f"Run started at: {RUN_START_UTC}")

## Load GLiNER2

In [ ]:
if USE_GPU_IF_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

if DEVICE == "cpu":
    print("WARNING: GPU is unavailable or disabled. CPU inference can be slow.")
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

model = load_gliner2_model(MODEL_NAME, device=DEVICE)
try:
    print("classify_text signature:", inspect.signature(model.classify_text))
except Exception as exc:
    print("Could not inspect classify_text signature:", exc)

## Quick start examples

In [ ]:
quick_examples = [
    "I still haven't received my new card after two weeks.",
    "I forgot my passcode and cannot access the app.",
    "Why was I charged twice for the same card transaction?",
    "Can I use Apple Pay with my card?",
    "I need to change my phone number.",
]
quick_label_texts = [
    "card_arrival",
    "passcode_forgotten",
    "cash_withdrawal",
    "apple_pay_or_google_pay",
    "edit_personal_details",
    "cash_withdrawal_card",
    "cash_withdrawal_not_recognised",
]

quick_variants = build_all_schema_variants(quick_label_texts, [PLAIN_LABEL])
quick_candidates = quick_variants[PLAIN_LABEL]["candidate_labels"]
quick_mapping = quick_variants[PLAIN_LABEL]["candidate_to_canonical"]

for text in quick_examples:
    row = predict_one_classification(
        model=model,
        text=text,
        candidate_labels=quick_candidates,
        candidate_to_canonical=quick_mapping,
        method_name=PLAIN_LABEL,
    )
    print("=" * 80)
    print("text:", text)
    print("predicted_candidate:", row["predicted_candidate"])
    print("predicted_canonical:", row["predicted_canonical"])
    print("confidence:", row["confidence"])
    print("parse_error:", row["parse_error"])
    print("raw_output:", row["raw_output"])

## Load Banking77

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
print("Split names:", list(dataset.keys()))
for split_name, split in dataset.items():
    print(f"{split_name}: {len(split)} examples; columns={split.column_names}")

if "test" not in dataset:
    raise ValueError("Dataset does not contain a test split.")

test_split = dataset["test"]
if "text" not in test_split.column_names:
    raise ValueError("Test split does not contain a text column.")
test_split = ensure_label_text_column(test_split)
if "label_text" not in test_split.column_names:
    raise ValueError("Test split does not contain label_text after validation.")

label_texts = unique_label_texts(test_split)
print(f"Number of labels: {len(label_texts)}")
print("First 10 labels:", label_texts[:10])

## Build evaluation subset

In [ ]:
def dataset_to_rows(split):
    rows = []
    for idx, row in enumerate(split):
        item = dict(row)
        item.setdefault("example_id", idx)
        rows.append(item)
    return rows


if MODE == "smoke":
    eval_examples = stratified_subset(
        test_split,
        per_label=1,
        seed=SEED,
        max_examples=SMOKE_N_EXAMPLES,
    )
elif MODE == "small":
    eval_examples = stratified_subset(
        test_split,
        per_label=SMALL_PER_LABEL,
        seed=SEED,
    )
elif MODE == "full":
    if not FULL_USE_ALL_TEST:
        raise ValueError("MODE='full' currently expects FULL_USE_ALL_TEST=True.")
    eval_examples = dataset_to_rows(test_split)
else:
    raise ValueError("MODE must be one of: smoke, small, full.")

example_ids = [example["example_id"] for example in eval_examples]
label_counts = pd.Series([example["label_text"] for example in eval_examples]).value_counts()
label_coverage = int(label_counts.shape[0])

print(f"Mode: {MODE}")
print(f"Evaluation examples: {len(eval_examples)}")
print(f"Label coverage: {label_coverage} / {len(label_texts)}")
print(label_counts.sort_index().head(10))

estimated_ensemble_input_calls = len(eval_examples) * len(ENSEMBLE_INPUT_METHODS)
estimated_all_single_schema_calls = len(eval_examples) * len(SINGLE_SCHEMA_METHODS)
print(
    f"Estimated model calls for B/C/D/E ensemble inputs: "
    f"{len(eval_examples)} examples x {len(ENSEMBLE_INPUT_METHODS)} variants = "
    f"{estimated_ensemble_input_calls}"
)
print(
    f"Estimated model calls including raw_label analysis: "
    f"{len(eval_examples)} examples x {len(SINGLE_SCHEMA_METHODS)} variants = "
    f"{estimated_all_single_schema_calls}"
)

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
(OUTPUT_PATH / "tables").mkdir(exist_ok=True)
(OUTPUT_PATH / "figures").mkdir(exist_ok=True)
(OUTPUT_PATH / "predictions").mkdir(exist_ok=True)

with (OUTPUT_PATH / "evaluated_example_ids.json").open("w", encoding="utf-8") as f:
    json.dump(example_ids, f, indent=2)
print("Saved evaluated example IDs.")

## Build schema variants

In [ ]:
schema_variants = build_all_schema_variants(label_texts, SINGLE_SCHEMA_METHODS)

for method_name, variant in schema_variants.items():
    candidates = variant["candidate_labels"]
    mapping = variant["candidate_to_canonical"]
    print(f"{method_name}: {len(candidates)} candidates, {len(mapping)} mappings")
    print("  sample:", candidates[:3])
    if len(candidates) != len(set(candidates)):
        raise ValueError(f"Duplicate candidates remain for {method_name}.")

schema_path = OUTPUT_PATH / "schema_variants.json"
with schema_path.open("w", encoding="utf-8") as f:
    json.dump(schema_variants, f, ensure_ascii=False, indent=2)
print(f"Saved schema variants to {schema_path}")

## Cache and manifest helpers

In [ ]:
def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=str(PROJECT_ROOT),
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        return None


def load_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=json_safe) + "\n")


def cache_manifest_matches(path, manifest):
    path = Path(path)
    if not path.exists():
        return False
    try:
        cached = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return cached == manifest


def save_method_cache(rows, method_name, manifest):
    pred_path = OUTPUT_PATH / "predictions" / f"{method_name}.jsonl"
    manifest_path = OUTPUT_PATH / "predictions" / f"{method_name}.manifest.json"
    save_jsonl(rows, pred_path)
    manifest_path.write_text(json.dumps(manifest, indent=2, default=json_safe), encoding="utf-8")
    return pred_path


def load_method_cache_if_valid(method_name, manifest):
    pred_path = OUTPUT_PATH / "predictions" / f"{method_name}.jsonl"
    manifest_path = OUTPUT_PATH / "predictions" / f"{method_name}.manifest.json"
    if FORCE_RERUN:
        return None
    if pred_path.exists() and cache_manifest_matches(manifest_path, manifest):
        print(f"Loading cached predictions for {method_name}: {pred_path}")
        return load_jsonl(pred_path)
    return None


BASE_MANIFEST = {
    "git_commit": get_git_commit(),
    "package_versions": environment_info,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": "test",
    "mode": MODE,
    "number_of_examples": len(eval_examples),
    "example_ids": example_ids,
    "device": DEVICE,
    "seed": SEED,
    "small_per_label": SMALL_PER_LABEL,
    "smoke_n_examples": SMOKE_N_EXAMPLES,
    "margin_threshold": MARGIN_THRESHOLD,
}

print(json.dumps({k: BASE_MANIFEST[k] for k in ["model_name", "dataset_name", "mode", "number_of_examples", "device"]}, indent=2))

## Run single-schema baselines

In [ ]:
predictions_by_method = {}

for method_name in SINGLE_SCHEMA_METHODS:
    variant = schema_variants[method_name]
    candidate_labels = list(variant["candidate_labels"])
    candidate_to_canonical = dict(variant["candidate_to_canonical"])
    method_manifest = {
        **BASE_MANIFEST,
        "method": method_name,
        "method_type": "single_schema",
        "candidate_labels_hash": stable_hash(candidate_labels),
        "candidate_to_canonical_hash": stable_hash(candidate_to_canonical),
    }

    cached = load_method_cache_if_valid(method_name, method_manifest)
    if cached is not None:
        predictions_by_method[method_name] = cached
        continue

    rows = []
    for example in tqdm(eval_examples, desc=method_name):
        row = predict_one_classification(
            model=model,
            text=example["text"],
            candidate_labels=candidate_labels,
            candidate_to_canonical=candidate_to_canonical,
            method_name=method_name,
            example_id=example["example_id"],
            gold_label=example["label_text"],
        )
        rows.append(row)

    save_method_cache(rows, method_name, method_manifest)
    predictions_by_method[method_name] = rows

print("Single-schema methods complete:", list(predictions_by_method))

## Run paraphrase ensembles

In [ ]:
ensemble_methods = {}

vote_manifest = {
    **BASE_MANIFEST,
    "method": VOTE_ENSEMBLE,
    "method_type": "ensemble",
    "input_methods": ENSEMBLE_INPUT_METHODS,
    "input_prediction_hash": stable_hash(
        {method: predictions_by_method[method] for method in ENSEMBLE_INPUT_METHODS}
    ),
}
cached = load_method_cache_if_valid(VOTE_ENSEMBLE, vote_manifest)
if cached is None:
    cached = majority_vote_ensemble(predictions_by_method)
    save_method_cache(cached, VOTE_ENSEMBLE, vote_manifest)
predictions_by_method[VOTE_ENSEMBLE] = cached
ensemble_methods[VOTE_ENSEMBLE] = "available"

coverage = confidence_coverage(predictions_by_method)
print(f"Confidence coverage across B/C/D/E: {coverage:.3f}")

if coverage == 0.0:
    print("Skipping confidence-based ensembles because no confidence values are available.")
    ensemble_methods[MEAN_CONFIDENCE_ENSEMBLE] = "unavailable: confidence missing"
    ensemble_methods[CONFIDENCE_MARGIN_FALLBACK] = "unavailable: confidence missing"
else:
    mean_manifest = {
        **BASE_MANIFEST,
        "method": MEAN_CONFIDENCE_ENSEMBLE,
        "method_type": "ensemble",
        "input_methods": ENSEMBLE_INPUT_METHODS,
        "input_prediction_hash": stable_hash(
            {method: predictions_by_method[method] for method in ENSEMBLE_INPUT_METHODS}
        ),
    }
    cached = load_method_cache_if_valid(MEAN_CONFIDENCE_ENSEMBLE, mean_manifest)
    if cached is None:
        cached = mean_confidence_ensemble(predictions_by_method)
        save_method_cache(cached, MEAN_CONFIDENCE_ENSEMBLE, mean_manifest)
    predictions_by_method[MEAN_CONFIDENCE_ENSEMBLE] = cached
    ensemble_methods[MEAN_CONFIDENCE_ENSEMBLE] = "available"

    fallback_manifest = {
        **BASE_MANIFEST,
        "method": CONFIDENCE_MARGIN_FALLBACK,
        "method_type": "ensemble",
        "input_methods": ENSEMBLE_INPUT_METHODS,
        "margin_threshold": MARGIN_THRESHOLD,
        "input_prediction_hash": stable_hash(
            {method: predictions_by_method[method] for method in ENSEMBLE_INPUT_METHODS}
        ),
    }
    cached = load_method_cache_if_valid(CONFIDENCE_MARGIN_FALLBACK, fallback_manifest)
    if cached is None:
        cached = confidence_margin_fallback(predictions_by_method, MARGIN_THRESHOLD)
        save_method_cache(cached, CONFIDENCE_MARGIN_FALLBACK, fallback_manifest)
    predictions_by_method[CONFIDENCE_MARGIN_FALLBACK] = cached
    ensemble_methods[CONFIDENCE_MARGIN_FALLBACK] = "available"

print("Ensemble status:")
print(json.dumps(ensemble_methods, indent=2))

## Compute metrics

In [ ]:
tables_dir = OUTPUT_PATH / "tables"
figures_dir = OUTPUT_PATH / "figures"

effective_latency_components = {
    VOTE_ENSEMBLE: ENSEMBLE_INPUT_METHODS,
    MEAN_CONFIDENCE_ENSEMBLE: ENSEMBLE_INPUT_METHODS,
    CONFIDENCE_MARGIN_FALLBACK: ENSEMBLE_INPUT_METHODS,
}
results_summary = method_metrics_frame(
    predictions_by_method,
    effective_latency_components=effective_latency_components,
)
per_label_metrics = per_label_metrics_frame(predictions_by_method)

results_summary_path = tables_dir / "results_summary.csv"
per_label_path = tables_dir / "per_label_metrics.csv"
results_summary.to_csv(results_summary_path, index=False)
per_label_metrics.to_csv(per_label_path, index=False)

display(results_summary.sort_values("macro_f1", ascending=False).reset_index(drop=True))
print(f"Saved {results_summary_path}")
print(f"Saved {per_label_path}")

## Analyze schema sensitivity

In [ ]:
schema_disagreement = schema_disagreement_frame(
    predictions_by_method,
    ENSEMBLE_INPUT_METHODS,
)
schema_disagreement_path = tables_dir / "schema_disagreement_examples.csv"
schema_disagreement.to_csv(schema_disagreement_path, index=False)

if schema_disagreement.empty:
    disagreement_rate = 0.0
else:
    disagreement_rate = float((~schema_disagreement["all_agree"]).mean())

print(f"Schema disagreement rate among B/C/D/E: {disagreement_rate:.3f}")
print("Unique predicted-label counts:")
display(schema_disagreement["unique_prediction_count"].value_counts().sort_index())
print(f"Saved {schema_disagreement_path}")
display(schema_disagreement[~schema_disagreement["all_agree"]].head(10))

## Error analysis

In [ ]:
ensemble_candidates = [
    method
    for method in [VOTE_ENSEMBLE, MEAN_CONFIDENCE_ENSEMBLE, CONFIDENCE_MARGIN_FALLBACK]
    if method in predictions_by_method
]
if ensemble_candidates:
    ensemble_scores = results_summary[results_summary["method"].isin(ensemble_candidates)]
    best_ensemble = ensemble_scores.sort_values("macro_f1", ascending=False).iloc[0]["method"]
else:
    best_ensemble = VOTE_ENSEMBLE

MAIN_SELECTED_METHOD = MEAN_CONFIDENCE_ENSEMBLE
SECONDARY_METHOD = VOTE_ENSEMBLE
comparison_methods = [
    method
    for method in [MAIN_SELECTED_METHOD, SECONDARY_METHOD]
    if method in predictions_by_method
]

print(f"Main selected method: {MAIN_SELECTED_METHOD}")
print(f"Secondary method: {SECONDARY_METHOD}")
print(f"Best observed ensemble for analysis: {best_ensemble}")

paired_summary = paired_summary_frame(
    predictions_by_method,
    baseline_method=PLAIN_LABEL,
    comparison_methods=comparison_methods,
)
paired_summary_path = tables_dir / "paired_analysis_summary.csv"
paired_summary.to_csv(paired_summary_path, index=False)

for comparison_method in comparison_methods:
    paired = paired_outcome_frame(
        predictions_by_method,
        baseline_method=PLAIN_LABEL,
        comparison_method=comparison_method,
    )
    paired.to_csv(tables_dir / f"paired_analysis_{comparison_method}.csv", index=False)

improved_examples, degraded_examples = compare_method_examples(
    predictions_by_method,
    baseline_method=PLAIN_LABEL,
    comparison_method=MAIN_SELECTED_METHOD,
)
improved_path = tables_dir / "improved_examples.csv"
degraded_path = tables_dir / "degraded_examples.csv"
improved_examples.to_csv(improved_path, index=False)
degraded_examples.to_csv(degraded_path, index=False)

all_confusions = []
for method_name, method_rows in predictions_by_method.items():
    method_confusions = confusion_pairs_frame(method_rows, top_n=50)
    if method_confusions.empty:
        continue
    method_confusions["method"] = method_name
    all_confusions.append(method_confusions)

confusion_pairs = (
    pd.concat(all_confusions, ignore_index=True)
    if all_confusions
    else pd.DataFrame(columns=["gold_label", "predicted_canonical", "count", "method"])
)
confusion_path = tables_dir / "confusion_pairs.csv"
confusion_pairs.to_csv(confusion_path, index=False)
plain_confusions = confusion_pairs[confusion_pairs["method"] == PLAIN_LABEL]

label_deltas = per_label_delta_frame(per_label_metrics, PLAIN_LABEL, best_ensemble)
label_deltas_path = tables_dir / "label_f1_deltas.csv"
label_deltas.to_csv(label_deltas_path, index=False)

plain_confidence = confidence_analysis_frame(predictions_by_method[PLAIN_LABEL])
plain_confidence_path = tables_dir / "plain_label_confidence_analysis.csv"
plain_confidence.to_csv(plain_confidence_path, index=False)

if CONFIDENCE_MARGIN_FALLBACK in predictions_by_method:
    fallback_frame = pd.DataFrame(predictions_by_method[CONFIDENCE_MARGIN_FALLBACK])
    fallback_rate = float(fallback_frame["fallback_used"].mean()) if not fallback_frame.empty else 0.0
    margin_path = tables_dir / "confidence_margin_distribution.csv"
    fallback_frame[["example_id", "gold_label", "predicted_canonical", "confidence_margin", "fallback_used"]].to_csv(margin_path, index=False)
else:
    fallback_rate = None

print("Paired analysis summary:")
display(paired_summary)
print(f"plain wrong -> {MAIN_SELECTED_METHOD} correct: {len(improved_examples)}")
print(f"plain correct -> {MAIN_SELECTED_METHOD} wrong: {len(degraded_examples)}")
print(f"Fallback rate: {fallback_rate}")
print("Top plain_label confusions:")
display(plain_confusions.head(15))
print("Largest F1 improvements:")
display(label_deltas.head(15))
print("Largest F1 degradations:")
display(label_deltas.sort_values("f1_delta").head(15))

## Visualizations

In [ ]:
figure_paths = []
figure_paths.append(
    plot_metric_bar(
        results_summary,
        "accuracy",
        figures_dir / "method_comparison_accuracy.png",
        "Method Comparison: Accuracy",
    )
)
figure_paths.append(
    plot_metric_bar(
        results_summary,
        "macro_f1",
        figures_dir / "method_comparison_macro_f1.png",
        "Method Comparison: Macro F1",
    )
)
figure_paths.append(
    plot_schema_disagreement_histogram(
        schema_disagreement,
        figures_dir / "schema_disagreement_histogram.png",
    )
)
figure_paths.append(
    plot_top_label_improvements(
        label_deltas,
        figures_dir / "top_15_label_improvements.png",
    )
)

top_pairs = plain_confusions.head(15)
confusion_labels = sorted(
    set(top_pairs["gold_label"].tolist()) | set(top_pairs["predicted_canonical"].tolist())
)
if confusion_labels:
    figure_paths.append(
        plot_confusion_matrix_for_pairs(
            predictions_by_method[PLAIN_LABEL],
            confusion_labels,
            figures_dir / "confusion_matrix_plain_label_top_pairs.png",
            "plain_label Confusion Matrix: Top Confused Labels",
        )
    )
    figure_paths.append(
        plot_confusion_matrix_for_pairs(
            predictions_by_method[best_ensemble],
            confusion_labels,
            figures_dir / f"confusion_matrix_{best_ensemble}_top_pairs.png",
            f"{best_ensemble} Confusion Matrix: Top Confused Labels",
        )
    )

print("Saved figures:")
for path in figure_paths:
    print(path)

## Save artifacts and run manifest

In [ ]:
RUN_END_UTC = datetime.now(timezone.utc).isoformat()
run_manifest = {
    **BASE_MANIFEST,
    "start_time_utc": RUN_START_UTC,
    "end_time_utc": RUN_END_UTC,
    "methods": list(predictions_by_method.keys()),
    "schema_variant_definitions": {
        RAW_LABEL: "candidate = original label_text",
        PLAIN_LABEL: "candidate = normalized label_text with underscores replaced by spaces",
        QUERY_ABOUT_LABEL: "candidate = question about {plain_label}",
        BANKING_REQUEST_LABEL: "candidate = banking support request about {plain_label}",
        CUSTOMER_INTENT_LABEL: "candidate = customer intent: {plain_label}",
        VOTE_ENSEMBLE: "majority vote over plain/query/banking/customer schema variants",
        MEAN_CONFIDENCE_ENSEMBLE: "mean confidence aggregation over plain/query/banking/customer schema variants",
        CONFIDENCE_MARGIN_FALLBACK: f"mean confidence if margin >= {MARGIN_THRESHOLD}, otherwise plain_label",
    },
    "confidence_coverage": coverage,
    "ensemble_status": ensemble_methods,
    "main_selected_method": MAIN_SELECTED_METHOD,
    "secondary_method": SECONDARY_METHOD,
    "best_observed_ensemble_for_analysis": best_ensemble,
    "artifacts": {
        "predictions_dir": str(OUTPUT_PATH / "predictions"),
        "tables_dir": str(tables_dir),
        "figures_dir": str(figures_dir),
        "paired_analysis_summary": str(tables_dir / "paired_analysis_summary.csv"),
        "schema_variants": str(schema_path),
        "evaluated_example_ids": str(OUTPUT_PATH / "evaluated_example_ids.json"),
    },
}

manifest_path = OUTPUT_PATH / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2, default=json_safe), encoding="utf-8")
print(f"Saved run manifest to {manifest_path}")

## Final short summary for presentation

In [ ]:
summary = results_summary.sort_values("macro_f1", ascending=False)
print("Experiment complete.")
print(f"Dataset: {DATASET_NAME} / test")
print(f"Mode: {MODE}")
print(f"Examples: {len(eval_examples)}")
print(f"Label coverage: {label_coverage} / {len(label_texts)}")
print(f"Schema disagreement rate B/C/D/E: {disagreement_rate:.3f}")
print("Top methods by macro F1:")
display(summary[["method", "accuracy", "macro_f1", "weighted_f1", "parse_failure_rate"]])
print("Key output files:")
for path in [
    OUTPUT_PATH / "run_manifest.json",
    tables_dir / "results_summary.csv",
    tables_dir / "per_label_metrics.csv",
    tables_dir / "schema_disagreement_examples.csv",
    tables_dir / "improved_examples.csv",
    tables_dir / "degraded_examples.csv",
    tables_dir / "confusion_pairs.csv",
]:
    print(path)